In [0]:
import requests

# Retrieve OAuth token
tenant_id = '1356bcf3-c075-43d5-a54c-ba71df07ff70'
API_APP_ID_URI = 'api://aeddc053-d47f-4352-9977-4313e0625905'

CLIENT_ID = dbutils.secrets.get(scope="apim_00010_1", key="client_id")
CLIENT_SECRET = dbutils.secrets.get(scope="apim_00010_1", key="client_secret")

proxies = {
    'http': 'http://awsproxy.int.corp.sun:8989',
    'https': 'http://awsproxy.int.corp.sun:8989',
}

token_endpoint = f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
token_headers = {
    "Content-Type": "application/x-www-form-urlencoded"
}
token_data = {
    "grant_type": "client_credentials",
    "resource": API_APP_ID_URI
}

# response = requests.post(url=token_endpoint, headers=token_headers, data=token_data, auth=(CLIENT_ID, CLIENT_SECRET), proxies=proxies)
response = requests.post(url=token_endpoint, headers=token_headers, data=token_data, auth=(CLIENT_ID, CLIENT_SECRET))
token = response.json()["access_token"]

In [0]:
import base64

def image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
    return encoded_string

In [0]:
ADI_APIM_BASE_URL = 'https://apim-nonprod-idp.azure-api.net/documentintelligence' #Add ADI APIM base URL here
MODEL_ID = 'prebuilt-layout'
MODEL_ID = 'prebuilt-read'

ADI_ANALYZE_URL = ADI_APIM_BASE_URL+'/documentModels/'+MODEL_ID+':analyze?_overload=analyzeDocument&api-version=2024-11-30&features=&locale=en-US&pages=1&outputContentFormat=markdown'

base64_string = image_to_base64("/Volumes/datascience_dev_bronze_sandbox/ds_document_deidentification/sessions/integration_test_004/original.png")

adi_input = {
    'base64Source': base64_string
}

adi_headers = {
    'Content-Type': 'application/json',
    'Authorization': 'Bearer '+token,
    'AppspaceId': 'A-007100'
}

In [0]:
adi_response = requests.post(ADI_ANALYZE_URL, json=adi_input, headers=adi_headers)

In [0]:
# Check response status
print(f"Status Code: {adi_response.status_code}")
print(f"Response Headers: {dict(adi_response.headers)}")
print(f"Response Text: {adi_response.text}")

# For 202 responses, get the operation location to poll for results
if adi_response.status_code == 202:
    operation_location = adi_response.headers.get('Operation-Location')
    print(f"Operation Location: {operation_location}")
else:
    # For other status codes, try to parse JSON if available
    if adi_response.text.strip():
        data = adi_response.json()
        print(f"\nResponse Data: {data}")
    else:
        print("\nResponse body is empty")

In [0]:
import time

# Extract operation location from 202 response
if adi_response.status_code == 202:
    operation_location = adi_response.headers.get('Operation-Location')
    
    if operation_location:
        print(f"Polling for results at: {operation_location}")
        
        # Poll the operation location until complete
        max_attempts = 30
        poll_interval = 2  # seconds
        
        for attempt in range(max_attempts):
            print(f"\nAttempt {attempt + 1}/{max_attempts}...")
            
            # Make GET request to operation location
            result_response = requests.get(
                url=operation_location,
                headers={'Authorization': f'Bearer {token}', 'AppspaceId': 'A-007100'}            
            )
            
            if result_response.status_code == 200:
                result_data = result_response.json()
                status = result_data.get('status')
                
                print(f"Status: {status}")
                
                if status == 'succeeded':
                    print("\nDocument analysis completed successfully!")
                    print(f"\nResult: {str(result_data)[:500]}...")
                    
                    # Store the final result
                    adi_result = result_data
                    break
                elif status == 'failed':
                    print(f"\n✗ Analysis failed: {result_data.get('error', 'Unknown error')}")
                    break
                elif status in ['running', 'notStarted']:
                    print(f"Still processing... waiting {poll_interval} seconds")
                    time.sleep(poll_interval)
                else:
                    print(f"Unknown status: {status}")
                    break
            else:
                print(f"Error polling results: {result_response.status_code}")
                print(f"Response: {result_response.text}")
                break
        else:
            print("\n✗ Timeout: Maximum polling attempts reached")
    else:
        print("✗ No Operation-Location header found in response")
else:
    print(f"Response status is {adi_response.status_code}, not 202")

## Using ADI Client

In [0]:
%pip install azure-ai-documentintelligence>=1.0.0 azure-core>=1.29.0
dbutils.library.restartPython()

In [0]:
import logging

logging.basicConfig(level=logging.INFO)

In [0]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.identity import ClientSecretCredential
from azure.core.pipeline.transport import RequestsTransport

tenant_id = '1356bcf3-c075-43d5-a54c-ba71df07ff70'
client_id = dbutils.secrets.get(scope="apim_00010_1", key="client_id")
client_secret =  dbutils.secrets.get(scope="apim_00010_1", key="client_secret")
endpoint = 'https://apim-nonprod-idp.azure-api.net'


proxies = {
    'http': 'http://awsproxy.int.corp.sun:8989',
    'https': 'http://awsproxy.int.corp.sun:8989',
}

transport = RequestsTransport(proxies=proxies)

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret,
    transport=transport,
)

client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=credential,
    headers={'AppspaceId': 'A-007100'},
    transport=transport
)

In [0]:
with open("/Volumes/datascience_dev_bronze_sandbox/ds_document_deidentification/sessions/integration_test_002/original.pdf", "rb") as f:
    print(f)

In [0]:
import time

start = time.time()

with open("/Volumes/datascience_dev_bronze_sandbox/ds_document_deidentification/sessions/integration_test_002/original.pdf", "rb") as f:
    pdf_bytes = f.read()

print("read:", time.time() - start)

start = time.time()

poller = client.begin_analyze_document(
    "prebuilt-layout",
    body=pdf_bytes,
    content_type="application/pdf"
)

print("submitted:", time.time() - start)

In [0]:
with open("/Volumes/datascience_dev_bronze_sandbox/ds_document_deidentification/sessions/integration_test_002/original.pdf", "rb") as f:
    poller = client.begin_analyze_document(
        "prebuilt-layout",
        body=f,
        content_type="application/pdf"
    )
print("Submitted to ADI")

result = poller.result()

print("Analysis finished")

In [0]:
result = poller.result()

In [0]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential


In [0]:
from azure.core.credentials import AccessToken
from azure.core.pipeline.policies import BearerTokenCredentialPolicy

# Create a custom token credential for Bearer authentication
class BearerTokenCredential:
    def __init__(self, token: str):
        self._token = token
    
    def get_token(self, *scopes, **kwargs):
        # Return the token with a far future expiration
        return AccessToken(self._token, 9999999999)

# Use base URL as endpoint, not the full analyze URL
ADI_APIM_BASE_URL = 'https://apim-nonprod-idp.azure-api.net/documentintelligence' #Add ADI APIM base URL here
MODEL_ID = 'prebuilt-layout'
MODEL_ID = 'prebuilt-read'

ADI_ANALYZE_URL = ADI_APIM_BASE_URL+'/documentModels/'+MODEL_ID+':analyze?_overload=analyzeDocument&api-version=2024-11-30&features=&locale=en-US&pages=1&outputContentFormat=markdown'

adi_client = DocumentIntelligenceClient(
    endpoint=ADI_APIM_BASE_URL,
    credential=BearerTokenCredential(token),
    headers={'AppspaceId': 'A-007100'}
)

print("ADI client created with Bearer token authentication")

In [0]:
import base64

def image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
    return encoded_string

In [0]:
filename = "/Volumes/datascience_dev_bronze_sandbox/ds_document_deidentification/sessions/integration_test_002/original.pdf"
base64_string = image_to_base64(filename)

print("Starting document analysis with ADI client...")

with open(filename, "rb") as f:
    poller = adi_client.begin_analyze_document(
        model_id="prebuilt-layout",
        body=f,
        content_type="application/pdf"
    )

print("Polling for results...")
result = poller.result()
print("✓ Document analysis completed successfully!")

In [0]:
# Display the results
print(f"Status: Success")
print(f"\nContent (first 500 chars):\n{result.content[:500]}...")
print(f"\nNumber of pages analyzed: {len(result.pages)}")

if result.pages:
    page = result.pages[0]
    print(f"Page dimensions: {page.width} x {page.height}")
    print(f"Number of lines: {len(page.lines) if page.lines else 0}")
    print(f"Number of words: {len(page.words) if page.words else 0}")

# Store result for later use
adi_result_client = result

In [0]:
# Extract text and words
full_text = result.content
words = []

for page in result.pages:
    for word in page.words:
        # Convert ADI polygon to bounding box
        bbox = self._polygon_to_bbox(word.polygon)
        words.append({
            "text": word.content,
            "confidence": word.confidence,
            "bounding_box": bbox
        })

return {
    "text": full_text,
    "words": words
}